In [3]:
import pandas as pd
import numpy as np

# Загрузка данных

In [4]:
orders     = pd.read_csv("data/olist_orders_dataset.csv")
items      = pd.read_csv("data/olist_order_items_dataset.csv")
products   = pd.read_csv("data/olist_products_dataset.csv")
customers  = pd.read_csv("data/olist_customers_dataset.csv")
cat_trans  = pd.read_csv("data/product_category_name_translation.csv")

# Задание 1

Слияние имен на разных языках

In [5]:
#Добвляем в таблицу продуктов столбец с именами на английском
products = products.merge(
    cat_trans,
    how="left"
)

# заменяем пропуски на unknown
products["product_category_name_english"] = (
    products["product_category_name_english"]
    .fillna("unknown")
)

Подзадача. Стоимость каждого уникального заказа с учетом цены товаров и доставки

In [6]:
order_agg = (
    items
    .groupby("order_id") #объединяем все предметы из одного заказа
    .agg(
        total_items_price=("price", "sum"),    # считаем общую сумму всех продуктов в заказе
        freight_value=("freight_value", "max"),# максимальная цена доставки 
    )
    #добавляем общую стоимость заказа
    .assign(total_order_value=lambda df: df["total_items_price"] + df["freight_value"])
    #.reset_index()
)
order_agg

,total_items_price,freight_value,total_order_value
order_id,,,
00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19
00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83
000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87
00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78
00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04
...,...,...,...
fffc94f6ce00a00581880bf54a75a037,299.99,43.41,343.40
fffcd46ef2263f404302a634eb57f7eb,350.00,36.53,386.53
fffce4705a9662cd70adb13d4a31832d,99.90,16.95,116.85


In [7]:
items

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72


Полное слияние таблиц

In [8]:
df = (
    orders
    .merge(order_agg, on="order_id", how="inner")
    .merge(customers, on="customer_id", how="left")
    .merge(
        items.drop_duplicates("order_id").drop("freight_value", axis=1),
        on="order_id", how="left"
    )
    .merge(products, on="product_id", how="left")
)
df

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,total_items_price,freight_value,...,price,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,29.99,8.72,...,29.99,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,118.70,22.76,...,118.70,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,159.90,19.22,...,159.90,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,45.00,27.20,...,45.00,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,19.90,8.72,...,19.90,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,stationery
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98661,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00,72.00,13.08,...,72.00,beleza_saude,50.0,1517.0,1.0,1175.0,22.0,13.0,18.0,health_beauty
98662,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00,174.90,20.10,...,174.90,bebes,52.0,828.0,4.0,4950.0,40.0,10.0,40.0,baby
98663,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00,205.99,65.02,...,205.99,eletrodomesticos_2,51.0,500.0,2.0,13300.0,32.0,90.0,22.0,home_appliances_2
98664,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00,359.98,40.59,...,179.99,informatica_acessorios,59.0,1893.0,1.0,6550.0,20.0,20.0,20.0,computers_accessories


Находим только доставленные заказы

In [9]:
df_delivered = df[df["order_status"] == "delivered"].copy()
df_delivered

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,total_items_price,freight_value,...,price,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,29.99,8.72,...,29.99,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,118.70,22.76,...,118.70,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,159.90,19.22,...,159.90,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,45.00,27.20,...,45.00,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,19.90,8.72,...,19.90,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,stationery
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98661,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00,72.00,13.08,...,72.00,beleza_saude,50.0,1517.0,1.0,1175.0,22.0,13.0,18.0,health_beauty
98662,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00,174.90,20.10,...,174.90,bebes,52.0,828.0,4.0,4950.0,40.0,10.0,40.0,baby
98663,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00,205.99,65.02,...,205.99,eletrodomesticos_2,51.0,500.0,2.0,13300.0,32.0,90.0,22.0,home_appliances_2
98664,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00,359.98,40.59,...,179.99,informatica_acessorios,59.0,1893.0,1.0,6550.0,20.0,20.0,20.0,computers_accessories


# Задание 2

Перевод времени в нужный формат + вычисление сегодняшней даты

In [10]:
df_delivered["order_purchase_timestamp"] = pd.to_datetime(
    df_delivered["order_purchase_timestamp"]
)
today = df_delivered["order_purchase_timestamp"].max() + pd.Timedelta(days=1)

вычисление RFM метик

In [11]:
rfm = (
    df_delivered
    .groupby("customer_unique_id")
    .agg(
        last_purchase=("order_purchase_timestamp", "max"),
        frequency=("order_id",              "nunique"),   # количество уникальных заказов
        monetary=("total_order_value",       "sum"),      # общая стоимость всех заказов пользователя
        customer_state=("customer_state",    "first"),    # для топ-5 штатов
    )
    .assign(recency=lambda df: (today - df["last_purchase"]).dt.days) #дней с последней покупки
    .drop(columns="last_purchase")
    .reset_index()
)
rfm["R"] = pd.qcut(
    rfm["recency"],
    q=4,
    labels=[4, 3, 2, 1]
)

# Вариант по ТЗ, но в таком случае лучшая метрика будет только у одного клиента из всех
# А это попахивает бредом
# Если использовать это то не получится выполнить задание с топ 5 штатов
#rfm["F"] = pd.cut(rfm["frequency"], bins=4, labels=[1, 2, 3, 4])

# Ручное разбиение
"""
1 заказ → F=1 (одноразовые)
2 → F=2 (вернулись 1 раз)
3 → F=3 (лояльные)
4+ → F=4 (лучшие)
"""
rfm["F"] = pd.cut(
    rfm["frequency"],
    bins=[0, 1, 2, 3, 100],
    labels=[1, 2, 3, 4]
)

rfm["M"] = pd.qcut(
    rfm["monetary"],
    q=4,
    labels=[1, 2, 3, 4]
)
rfm.F.value_counts()

F
1    90557
2     2573
3      181
4       47
Name: count, dtype: int64

Суммируем RFM счет и находим чемпионов

In [22]:
rfm["rfm_score"] = (rfm["R"].astype(str) +
                    rfm["F"].astype(str) +
                    rfm["M"].astype(str))

champions = rfm[rfm["rfm_score"] == "444"] #Таблица где есть только самые лучшие клиенты
top5_states = (
    champions
    .groupby("customer_state")
    .size() # Просто считает количество строк в каждой группе
    .sort_values(ascending=False) # Сортировка по убыванию
    .head(5)
)

In [23]:
print("Лучшие штаты по количеству чемпионов")
top5_states

Лучшие штаты по количеству чемпионов


customer_state
SP    8
RJ    4
RS    4
PE    2
PR    1
dtype: int64

# Задание 3

Строим таблицу с категориями товаров

In [35]:
#Склеиваем таблицу только из нужных столбцов
items_cat = (
    items[["order_id", "product_id", "price"]]
    .merge(products[["product_id", "product_category_name_english"]],
           on="product_id", how="left")
    .merge(orders[["order_id", "customer_id", "order_status"]],
           on="order_id", how="left")
    .merge(customers[["customer_id", "customer_state"]],
           on="customer_id", how="left")
)

#Оставляем только доставленные товары и исправляем имена
items_cat = items_cat[items_cat["order_status"] == "delivered"].copy()

items_cat

,order_id,product_id,price,product_category_name_english,customer_id,order_status,customer_state
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,58.90,cool_stuff,3ce436f183e68e07877b285a838db11a,delivered,RJ
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,239.90,pet_shop,f6dd3ec061db4e3987629fe6b26e5cce,delivered,SP
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,199.00,furniture_decor,6489ae5e4333f3693df5ad4372dab6d3,delivered,MG
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,12.99,perfumery,d4eb9395c8c0431ee92fce09860c5a06,delivered,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,199.90,garden_tools,58dbd0b2d70206bf40e62cd34e84d795,delivered,SP
...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,4aa6014eceb682077f9dc4bffebc05b0,299.99,housewares,b51593916b4b8e0d6f66f2ae24f2673d,delivered,MA
112646,fffcd46ef2263f404302a634eb57f7eb,32e07fd915822b0765e448c4dd74c828,350.00,computers_accessories,84c5d4fbaf120aae381fad077416eaa0,delivered,PR
112647,fffce4705a9662cd70adb13d4a31832d,72a30483855e2eafc67aee5dc2560482,99.90,sports_leisure,29309aa813182aaddc9b259e31b870e6,delivered,SP
112648,fffe18544ffabc95dfada21779c9644f,9c422a519119dcad7575db5af1ba540e,55.99,computers_accessories,b5e6afd5a41800fdf401e0272ca74655,delivered,SP


Находим 10 самых продаваемых категорий в стране

In [ ]:
top10_cats = (
    items_cat
    .groupby("product_category_name_english")["price"]
    .sum()
    .nlargest(10)
    .index
)

#оставляем в таблице только те товары которые в топ 10
items_top10 = items_cat[items_cat["product_category_name_english"].isin(top10_cats)]
items_top10

,order_id,product_id,price,product_category_name_english,customer_id,order_status,customer_state
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,58.90,cool_stuff,3ce436f183e68e07877b285a838db11a,delivered,RJ
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,199.00,furniture_decor,6489ae5e4333f3693df5ad4372dab6d3,delivered,MG
5,00048cc3ae777c65dbb7d2a0634bc1ea,ef92defde845ab8450f9d70c526ef70f,21.90,housewares,816cbea969fe5b689b39cfc97a506742,delivered,MG
8,0005a1a1728c9d785b8e2b08b904576c,310ae3c140ff94b03219ad0adc3c778f,145.95,health_beauty,16150771dfd4776261284213b89c304e,delivered,SP
10,00061f2a7bc09da83e415a52dc8a4af1,d63c1011f49d98b976c352955b1c4bea,59.99,health_beauty,c6fc061d86fab1e2b2eac259bac71a49,delivered,SP
...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,4aa6014eceb682077f9dc4bffebc05b0,299.99,housewares,b51593916b4b8e0d6f66f2ae24f2673d,delivered,MA
112646,fffcd46ef2263f404302a634eb57f7eb,32e07fd915822b0765e448c4dd74c828,350.00,computers_accessories,84c5d4fbaf120aae381fad077416eaa0,delivered,PR
112647,fffce4705a9662cd70adb13d4a31832d,72a30483855e2eafc67aee5dc2560482,99.90,sports_leisure,29309aa813182aaddc9b259e31b870e6,delivered,SP
112648,fffe18544ffabc95dfada21779c9644f,9c422a519119dcad7575db5af1ba540e,55.99,computers_accessories,b5e6afd5a41800fdf401e0272ca74655,delivered,SP


Строим сводную таблицу

In [40]:
pivot = items_top10.pivot_table(
    values="price", # значения внутри таблицы
    index="customer_state", # что является строками
    columns="product_category_name_english", # что является столбцами
    aggfunc="sum", # Функция агрегации. Суммируем данные по штатам
    fill_value=0,  # Замена для пустых значений
    )
pivot

product_category_name_english,auto,bed_bath_table,computers_accessories,cool_stuff,furniture_decor,health_beauty,housewares,sports_leisure,toys,watches_gifts
customer_state,,,,,,,,,,
AC,540.98,567.70,1238.50,99.99,1316.44,1386.58,513.06,1677.46,234.79,1389.60
AL,4578.54,1930.14,7605.61,2401.04,3588.49,12681.26,600.36,3697.78,1053.07,11605.02
AM,723.90,681.20,1815.74,638.88,215.28,2776.03,408.69,1362.60,721.56,1755.35
AP,1216.39,669.50,2050.03,314.00,270.80,1380.58,490.49,812.69,297.84,1639.80
BA,29112.71,24949.53,32635.83,20142.26,23121.17,49955.12,16585.98,37010.06,14350.14,45631.52
CE,11806.30,6800.79,11551.97,12005.41,8507.34,31544.50,6958.06,10360.72,7793.31,29025.35
DF,17258.54,16147.72,23853.44,12095.99,12020.81,29542.82,12842.46,22467.28,11336.86,32292.27
ES,11831.37,22589.65,14424.18,10909.83,12912.72,20038.50,11509.46,20086.20,10458.46,28074.00
GO,11309.26,22285.61,13828.33,17492.22,12032.00,27971.30,14733.30,19231.55,7809.07,31486.93


Перевод значений в процентны. Какой процент от выручки конкретного штата состоявляет товар

In [41]:
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
#pivot.sum считает сумму всех значений в строке
#pivot.div делит каждое значение в строке на сумму

pivot_pct

product_category_name_english,auto,bed_bath_table,computers_accessories,cool_stuff,furniture_decor,health_beauty,housewares,sports_leisure,toys,watches_gifts
customer_state,,,,,,,,,,
AC,6.034289,6.332333,13.814681,1.115325,14.684053,15.466420,5.722859,18.711002,2.618933,15.500106
AL,9.204703,3.880356,15.290329,4.827054,7.214305,25.494423,1.206965,7.434022,2.117093,23.330749
AM,6.522074,6.137363,16.359153,5.756075,1.939594,25.011014,3.682147,12.276527,6.500992,15.815061
AP,13.305338,7.323247,22.424011,3.434652,2.962114,15.101311,5.365167,8.889514,3.257888,17.936759
BA,9.919344,8.500856,11.119748,6.862913,7.877894,17.020813,5.651210,12.610145,4.889410,15.547667
CE,8.658581,4.987608,8.472059,8.804606,6.239168,23.134311,5.102947,7.598412,5.715508,21.286800
DF,9.090227,8.505148,12.563819,6.371066,6.331468,15.560466,6.764238,11.833717,5.971225,17.008626
ES,7.265892,13.872778,8.858191,6.699955,7.929972,12.306063,7.068201,12.335356,6.422760,17.240832
GO,6.347114,12.507388,7.760895,9.817186,6.752738,15.698377,8.268793,10.793353,4.382697,17.671459


Поиск аномалий health beauty

In [ ]:
#Среднее значение
mean_val = pivot_pct["health_beauty"].mean()

#Штат с наибольшим процентом и его значние
anomaly_state = pivot_pct["health_beauty"].idxmax()
anomaly_val   = pivot_pct["health_beauty"].max()

print(f"Средняя доля health beauty: {mean_val:.2f}%")
print(f"Штат с наибольшей долей health beauty: {anomaly_state}")
print(f"Отклонение от среднего: {(anomaly_val - mean_val):.2f} пп")

Средняя доля health beauty: 18.27%
Штат с наибольшей долей health beauty: RN
Отклонение от среднего: 9.83 пп


# Задание 4

Создаем новую таблицу для кагорты и находим первый месяц покупки

In [44]:
cohort_df = df_delivered[["customer_unique_id", "order_id",
                            "order_purchase_timestamp"]].copy()

cohort_df["cohort_month"] = (
    cohort_df
    .groupby("customer_unique_id")["order_purchase_timestamp"]
    .transform("min") # Для каждого покупателя оставляем только дату первой покупки
    .dt.to_period("M")# Обрезаем значение даты до месяца
)
cohort_df

,customer_unique_id,order_id,order_purchase_timestamp,cohort_month
0,7c396fd4830fd04220f754e42b4e5bff,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-09
1,af07308b275d755c9edb36a90c618231,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,2018-07
2,3a653a41f6f9fc3d2a113cf8398680e8,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,2018-08
3,7c142cf63193a1473d2e66489a9ae977,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,2017-11
4,72632f0f9dd73dfee390c9b22eb56dd6,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,2018-02
...,...,...,...,...
98661,6359f309b166b0196dbf7ad2ac62bb5a,9c5dedf39a927c1b2549525ed64a053c,2017-03-09 09:54:05,2017-03
98662,da62f9e57a76d978d02ab5362c509660,63943bddc261676b46f01ca7ac2f7bd8,2018-02-06 12:58:58,2018-02
98663,737520a9aad80b3fbbdad19b66b37b30,83c1379a015df1e13d02aae0204711ab,2017-08-27 14:46:43,2017-08
98664,5097a5312c8b157bb7be58ae360ef43c,11c177c8e97725db2631073c19f07b62,2018-01-08 21:28:27,2018-01


Вычисляем Month Index

In [53]:
order_period = cohort_df["order_purchase_timestamp"].dt.to_period("M")
cohort_df["month_index"] = ((
    order_period - cohort_df["cohort_month"])
    .apply(lambda x: x.n) # Из <n*MonthEnds> извлекаем только число месяцев
)
cohort_df

,customer_unique_id,order_id,order_purchase_timestamp,cohort_month,month_index
0,7c396fd4830fd04220f754e42b4e5bff,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-09,1
1,af07308b275d755c9edb36a90c618231,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,2018-07,0
2,3a653a41f6f9fc3d2a113cf8398680e8,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,2018-08,0
3,7c142cf63193a1473d2e66489a9ae977,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,2017-11,0
4,72632f0f9dd73dfee390c9b22eb56dd6,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,2018-02,0
...,...,...,...,...,...
98661,6359f309b166b0196dbf7ad2ac62bb5a,9c5dedf39a927c1b2549525ed64a053c,2017-03-09 09:54:05,2017-03,0
98662,da62f9e57a76d978d02ab5362c509660,63943bddc261676b46f01ca7ac2f7bd8,2018-02-06 12:58:58,2018-02,0
98663,737520a9aad80b3fbbdad19b66b37b30,83c1379a015df1e13d02aae0204711ab,2017-08-27 14:46:43,2017-08,0
98664,5097a5312c8b157bb7be58ae360ef43c,11c177c8e97725db2631073c19f07b62,2018-01-08 21:28:27,2018-01,0


Построение кагортной таблицы

In [68]:
retention_abs = (
    cohort_df
    .groupby(["cohort_month", "month_index"])["customer_unique_id"] # обедияем по  месяцу и индексу
    .nunique() #считаем количество уникальных клиентов
    .reset_index() #После группировки месяца и инледсы месяцев стали индексами. Исправляем
    .pivot_table(index="cohort_month", 
                 columns="month_index",
                 values="customer_unique_id", 
                 fill_value=0) # строим сводную таблицу 
)
retention_abs

month_index,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,19,20
cohort_month,,,,,,,,,,,,,,,,,,,,
2016-09,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2016-10,262.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,2.0,2.0
2016-12,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2017-01,717.0,2.0,2.0,1.0,3.0,1.0,3.0,1.0,1.0,0.0,3.0,1.0,5.0,3.0,1.0,1.0,2.0,3.0,1.0,0.0
2017-02,1628.0,3.0,5.0,2.0,7.0,2.0,4.0,3.0,2.0,3.0,2.0,5.0,2.0,3.0,2.0,1.0,1.0,3.0,0.0,0.0
2017-03,2503.0,11.0,9.0,10.0,9.0,4.0,4.0,8.0,8.0,2.0,9.0,3.0,5.0,3.0,4.0,6.0,2.0,3.0,0.0,0.0
2017-04,2256.0,14.0,5.0,4.0,6.0,6.0,8.0,7.0,7.0,4.0,6.0,2.0,1.0,1.0,2.0,2.0,3.0,0.0,0.0,0.0
2017-05,3451.0,16.0,16.0,10.0,10.0,11.0,14.0,5.0,9.0,9.0,9.0,12.0,8.0,1.0,6.0,7.0,0.0,0.0,0.0,0.0
2017-06,3037.0,15.0,12.0,13.0,9.0,12.0,11.0,7.0,4.0,6.0,9.0,11.0,5.0,5.0,7.0,0.0,0.0,0.0,0.0,0.0


Доп задание. Перевод в проценты

In [72]:
cohort_sizes = retention_abs[0]  # размер когорты в Month 0
retention_pct = retention_abs.div(cohort_sizes, axis=0) * 100
retention_pct

month_index,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,19,20
cohort_month,,,,,,,,,,,,,,,,,,,,
2016-09,100.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2016-10,100.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.381679,0.000000,0.000000,0.381679,0.000000,0.381679,0.000000,0.381679,0.000000,0.381679,0.000000,0.381679,0.763359,0.763359
2016-12,100.0,100.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2017-01,100.0,0.278940,0.278940,0.139470,0.418410,0.139470,0.418410,0.139470,0.139470,0.000000,0.418410,0.139470,0.697350,0.418410,0.139470,0.139470,0.278940,0.418410,0.139470,0.000000
2017-02,100.0,0.184275,0.307125,0.122850,0.429975,0.122850,0.245700,0.184275,0.122850,0.184275,0.122850,0.307125,0.122850,0.184275,0.122850,0.061425,0.061425,0.184275,0.000000,0.000000
2017-03,100.0,0.439473,0.359569,0.399521,0.359569,0.159808,0.159808,0.319616,0.319616,0.079904,0.359569,0.119856,0.199760,0.119856,0.159808,0.239712,0.079904,0.119856,0.000000,0.000000
2017-04,100.0,0.620567,0.221631,0.177305,0.265957,0.265957,0.354610,0.310284,0.310284,0.177305,0.265957,0.088652,0.044326,0.044326,0.088652,0.088652,0.132979,0.000000,0.000000,0.000000
2017-05,100.0,0.463634,0.463634,0.289771,0.289771,0.318748,0.405680,0.144886,0.260794,0.260794,0.260794,0.347725,0.231817,0.028977,0.173863,0.202840,0.000000,0.000000,0.000000,0.000000
2017-06,100.0,0.493908,0.395127,0.428054,0.296345,0.395127,0.362200,0.230491,0.131709,0.197563,0.296345,0.362200,0.164636,0.164636,0.230491,0.000000,0.000000,0.000000,0.000000,0.000000
